In [ ]:
import sentencepiece as spm

import os
import unicodedata
import zipfile


def normalizes(text):
    text = unicodedata.normalize("NFKC", text.strip().lower())
    return eng.lower().strip(), hin.lower().strip()

text_pairs = []
with open("hin.txt", "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        eng , hin = normalizes(line)
        text_pairs.append((eng, hin))

en_corpus_path = "en_corpus.txt"
hi_corpus_path = "hi_corpus.txt"

with open(en_corpus_path, "w", encoding="utf-8") as f_en, \
     open(hi_corpus_path, "w", encoding="utf-8") as f_hi:
    for eng, hin in text_pairs:
        f_en.write(eng + "\n")
        f_hi.write(hin + "\n")
vocab_size = 9999999999999

if os.path.exists("en_model.model"):
    print("loading existing English SentencePiece model... ")
    en_sp = spm.SentencePieceProcessor(model_file="en_model.model")
else:
    print("Training English SentencePiece model...")
    spm.SentencePieceTrainer.Train(
        input= en_corpus_path, 
        model_prefix = "en_model",
        vocab_size = vocab_size, 
        model_type = "unigram", 
        user_defined_symbols=["[start]", "[end]"],  # Control tokens
        pad_id=0,
        unk_id=1,
        bos_id=-1,  # Disabled native BOS/EOS since we added custom ones
        eos_id=-1
    )
    en_sp = spm.SentencePieceProcessor(model_file="en_model.model")

if os.path.exists("hi_model.model"):
    print("Loading existing Hindi SentencePiece model...")
    hi_sp = spm.SentencePieceProcessor(model_file="hi_model.model")
else:
    print("Training Hindi SentencePiece model...")
    spm.SentencePieceTrainer.train(
        input=hi_corpus_path,
        model_prefix="hi_model",
        vocab_size=vocab_size,
        model_type="unigram",
        user_defined_symbols=["[start]", "[end]"],
        pad_id=0,
        unk_id=1,
        bos_id=-1,
        eos_id=-1
    )
    hi_sp = spm.SentencePieceProcessor(model_file="hi_model.model")

# Optional: Clean up the temporary corpus files from disk
# os.remove(en_corpus_path)
# os.remove(hi_corpus_path)



loading existing English SentencePiece model... 
Loading existing Hindi SentencePiece model...


In [24]:
import torch 
from torch.utils.data import Dataset, DataLoader

class data(Dataset):
    def __init__(self, textpari, en_sp, hi_sp):
        self.textpairs = textpari
        self.ensp = en_sp
        self.hisp = hi_sp

    def __len__(self):
        return len(self.textpairs)
    def __getitem__(self, index):
        engtxt, hitxt = self.textpairs[index]

        enids = self.ensp.encode(engtxt, out_type= int )     
        hiids = self.hisp.encode(hitxt , out_type=int)
        
        start_id = self.hisp.piece_to_id("[start]")
        end_id = self.hisp.piece_to_id("[end]")

        hi_input = [start_id] + hiids
        hi_tgt = hiids + [end_id]
        return (
            torch.tensor(enids, dtype=torch.long),
            torch.tensor(hi_input, dtype=torch.long),
            torch.tensor(hi_tgt, dtype=torch.long),)
def generate_mask(batch):
    en_batch , hiin_batch, hitgt_batch = [], [], []
    for en , hiin, hitgt in batch:
        en_batch.append(en)
        hiin_batch.append(hiin)
        hitgt_batch.append(hitgt)

        en_batch=torch.nn.utils.rnn.pad_sequence(en_batch, batch_first=True, padding_value=0)    
        hiin_batch=torch.nn.utils.rnn.pad_sequence(hiin_batch, batch_first=True, padding_value=0)    
        hitgt_batch=torch.nn.utils.rnn.pad_sequence(hitgt_batch, batch_first=True, padding_value=0)  

        return en_batch, hiin_batch, hitgt_batch
Dataset = data(textpari=text_pairs, en_sp=en_sp, hi_sp=hi_sp)
DataLoader = DataLoader(Dataset, batch_size=32, shuffle=True, collate_fn=generate_mask)


In [26]:
src_vocab_size = 2079
tgt_vocab_size = 2079
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

model = Badassatron(src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_length, dropout=dropout)

criterion = torch.nn.CrossEntropyLoss(ignore_index=0) # Ignore padding token (id=0) in loss calculation
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")#we don't have cpu (cry cry emoji)(motivation emoji hund me apna kuch banwanga with less comp power)

model.train()



for ep in range(10):
    total_loss = 0
    for src, tgt_input, tgt_output in DataLoader:
        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_output = tgt_output.to(device)

        optimizer.zero_grad()
        output = model(
            src,
            tgt_input

        )
        loss = criterion(output.reshape(-1, src_vocab_size), tgt_output.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {ep+1} Complete. Average Loss: {total_loss / len(DataLoader):.4f}")
        
import torch

# Save just the weights
model_save_path = "tranlator_en_hi.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved successfully to {model_save_path}")
        

Epoch 1 Complete. Average Loss: 5.9161
Epoch 2 Complete. Average Loss: 5.3706
Epoch 3 Complete. Average Loss: 5.1794
Epoch 4 Complete. Average Loss: 5.3709
Epoch 5 Complete. Average Loss: 4.9631
Epoch 6 Complete. Average Loss: 5.0861
Epoch 7 Complete. Average Loss: 4.8612
Epoch 8 Complete. Average Loss: 4.8396
Epoch 9 Complete. Average Loss: 4.6510
Epoch 10 Complete. Average Loss: 4.6862
Model weights saved successfully to tranlator_en_hi.pth


In [ ]:
# -----------------------------------------transformer with kv cache 
import math
import torch
import torch.nn as nn


class Multiheadattention(nn.Module):
    def __init__(self, dmodel, num_heads):
        super().__init__()
        assert dmodel % num_heads == 0

        self.dmodel = dmodel
        self.num_heads = num_heads
        self.d_k = dmodel // num_heads

        self.w_q = nn.Linear(dmodel, dmodel)
        self.w_k = nn.Linear(dmodel, dmodel)
        self.w_v = nn.Linear(dmodel, dmodel)
        self.w_o = nn.Linear(dmodel, dmodel)

    def scaler_product_attenion(self, Q, K, V, mask=None):
        atten_score = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            atten_score = atten_score.masked_fill(mask == 0, -1e9)
        atten_prob = torch.softmax(atten_score, dim=-1)
        output = torch.matmul(atten_prob, V)
        return output

    def splits_head(self, x):
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_head(self, x):
        batch_size, _, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.dmodel)

    def forward(self, Q, K, V, mask=None, past_kv=None, kv_override=None):
        """
        past_kv:    optional (K_cached, V_cached) tensors of shape
                    (batch, num_heads, past_len, d_k). New K/V computed from
                    K, V args are concatenated onto these (self-attention case).
        kv_override: optional (K, V) tensors to use directly instead of
                    projecting K, V args at all (cross-attention case, where
                    encoder K/V are computed once and reused every step).

        Returns (output, present_kv) where present_kv = (K_used, V_used),
        which the caller can feed back in as past_kv / kv_override on the
        next step.
        """
        Qh = self.splits_head(self.w_q(Q))

        if kv_override is not None:
            Kh, Vh = kv_override
        else:
            Kh = self.splits_head(self.w_k(K))
            Vh = self.splits_head(self.w_v(V))
            if past_kv is not None:
                past_k, past_v = past_kv
                Kh = torch.cat([past_k, Kh], dim=2)
                Vh = torch.cat([past_v, Vh], dim=2)

        atten_output = self.scaler_product_attenion(Qh, Kh, Vh, mask)
        output = self.w_o(self.combine_head(atten_output))
        present_kv = (Kh, Vh)
        return output, present_kv


class PositionWiseFF(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.relu = nn.GELU()

    def forward(self, x):
        return self.ff2(self.relu(self.ff1(x)))


class PositionEncoding(nn.Module):
    def __init__(self, d_model, max_seqlen):
        super(PositionEncoding, self).__init__()

        self.dropout = nn.Dropout(0.2)
        pe = torch.zeros(max_seqlen, d_model)

        postion = torch.arange(0, max_seqlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(postion * div_term)
        pe[:, 1::2] = torch.cos(postion * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x, start_pos=0):
        """
        start_pos lets us position-encode a single incoming token correctly
        when it's not the start of the sequence — this matters once you're
        feeding the model one cached token at a time instead of the whole
        sequence at once.
        """
        seq_len = x.size(1)
        x = x + self.pe[:, start_pos:start_pos + seq_len]
        return self.dropout(x)


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.attention = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.ff = PositionWiseFF(d_model=d_model, d_ff=d_ff)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        atten_out, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(atten_out))
        ffoutput = self.ff(x)
        out = self.norm2(x + self.dropout(ffoutput))
        return out


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, droupout):
        super(Decoder, self).__init__()
        self.atten = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.crossatten = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = PositionWiseFF(d_model=d_model, d_ff=d_ff)
        self.dropout = nn.Dropout(droupout)

    def forward(self, x, encoder_out, src_mask, tgt_mask, self_past_kv=None, cross_kv=None):
        """
        self_past_kv: cached self-attention K/V from previous decoding steps
                      (None on the very first step / during full training pass).
        cross_kv:     cached cross-attention K/V (computed once from
                      encoder_out). Pass None the first time this layer sees
                      encoder_out; pass the returned cross_present_kv back in
                      on every subsequent step so the encoder K/V projections
                      aren't recomputed each time.
        """
        atten_out, self_present_kv = self.atten(x, x, x, tgt_mask, past_kv=self_past_kv)
        x = self.norm1(x + self.dropout(atten_out))

        atten2, cross_present_kv = self.crossatten(
            x, encoder_out, encoder_out, src_mask, kv_override=cross_kv
        )
        x = self.norm2(x + self.dropout(atten2))

        ff_x = self.ff(x)
        output = self.norm3(x + self.dropout(ff_x))
        return output, self_present_kv, cross_present_kv


class Badassatron(nn.Module):
    def __init__(self, src_vocab_size, src_tgt_size, dmodel, numheads, dff, numlayers, max_lenseq, dropout):
        super().__init__()
        self.encoder_emb = nn.Embedding(src_vocab_size, dmodel)
        self.decoder_emb = nn.Embedding(src_tgt_size, dmodel)
        self.postion_enc = PositionEncoding(d_model=dmodel, max_seqlen=max_lenseq)

        self.encoder_layer = nn.ModuleList([Encoder(dmodel, numheads, dff, dropout) for _ in range(numlayers)])
        self.decoder_layer = nn.ModuleList([Decoder(dmodel, numheads, dff, dropout) for _ in range(numlayers)])

        self.fc = nn.Linear(dmodel, src_tgt_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_len = tgt.size(1)The NDA family will work together for the progress of Bihar.

        nopeak = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        """Standard teacher-forced training forward pass. No caching — the
        whole target sequence is available up front, so there's nothing to
        cache across steps."""
        srcmask, tgtmask = self.generate_mask(src, tgt)
        src_emb = self.dropout(self.postion_enc(self.encoder_emb(src)))
        tgt_emb = self.dropout(self.postion_enc(self.decoder_emb(tgt)))

        enc_output = src_emb
        for enc_layer in self.encoder_layer:
            enc_output = enc_layer(enc_output, srcmask)

        dec_output = tgt_emb
        for dec_layer in self.decoder_layer:
            dec_output, _, _ = dec_layer(dec_output, enc_output, srcmask, tgtmask)
        output = self.fc(dec_output)

        return output

    @torch.no_grad()
    def generate(self, src, start_token_id, end_token_id, max_len=50):
        """
        Incremental (autoregressive) decoding with KV caching.

        At each step we only run the newest token through the decoder — its
        query attends against cached K/V from all previous steps (self-
        attention) and against the encoder's K/V, computed once and reused
        every step (cross-attention). This avoids recomputing attention over
        the whole growing sequence at every step, which is what makes cached
        generation fast.
        """
        self.eval()
        device = src.device
        batch_size = src.size(0)

        # --- Encoder runs once, no caching needed (not autoregressive) ---
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        enc_output = self.dropout(self.postion_enc(self.encoder_emb(src)))
        for enc_layer in self.encoder_layer:
            enc_output = enc_layer(enc_output, src_mask)

        # --- Decoder: one token at a time, cache grows each step ---
        tgt = torch.full((batch_size, 1), start_token_id, dtype=torch.long, device=device)
        num_layers = len(self.decoder_layer)
        self_kv_caches = [None] * num_layers   # grows every step
        cross_kv_caches = [None] * num_layers  # computed once, reused every step

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(max_len):
            cur_input = tgt[:, -1:]  # only the newest token needs a forward pass
            tgt_emb = self.dropout(
                self.postion_enc(self.decoder_emb(cur_input), start_pos=step)
            )

            dec_output = tgt_emb
            for i, dec_layer in enumerate(self.decoder_layer):
                dec_output, self_kv_caches[i], cross_kv_caches[i] = dec_layer(
                    dec_output,
                    enc_output,
                    src_mask,
                    None,  # no causal mask needed: a single query position
                           # can only attend to past+current keys anyway
                    self_past_kv=self_kv_caches[i],
                    cross_kv=cross_kv_caches[i],
                )

            logits = self.fc(dec_output[:, -1, :])
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            tgt = torch.cat([tgt, next_token], dim=1)

            finished = finished | (next_token.squeeze(-1) == end_token_id)
            if finished.all():
                break

        return tgt